In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable


In [0]:
%run /Workspace/Users/vaishnavikumbhakarna467@gmail.com/Fast-Moving-Consumer-Good-Databricks/01_setup/utilities

In [0]:
print(bronze_schema, silver_schema, gold_schema)

In [0]:
dbutils.widgets.text("catalog","fmcg","catalog")
dbutils.widgets.text("data_source","customers","Data Source")

In [0]:
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")


base_path = f's3://sportsbar-vk-01/{data_source}/*.csv'
print(base_path)

In [0]:
df = (
    spark.read.format('csv')
    .option('header',True)
    .option('inferschema',True)
    .load(base_path)
    .withColumn('read_timestamp',F.current_timestamp())
    .select("*","_metadata.file_name","_metadata.file_size")
)
display(df.limit(10))

In [0]:
df.write\
    .format('delta') \
    .option("delta.enableChangeDataFeed", "true") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")

In [0]:
df.printSchema()

**Silver Processing**

In [0]:
df_bronze = spark.sql(f"SELECT * FROM {catalog}.{bronze_schema}.{data_source};")
df_bronze.show(5)

In [0]:
df_bronze.printSchema()

In [0]:
print("Rows before dropping duplicates: ",df_bronze.count())
df_silver = df_bronze.dropDuplicates(["customer_id"])
print("Rows After dropping duplicates: ",df_silver.count())

In [0]:
# to check the customers having leading spaces in their names

display(
    df_silver.filter(F.col("customer_name") != F.trim(F.col('customer_name')))
)

In [0]:
df_silver = df_silver.withColumn(
    "customer_name",
    F.trim(F.col("customer_name")))

In [0]:
display(
    df_silver.filter(F.col("customer_name") != F.trim(F.col('customer_name')))
)

In [0]:
#  cities name should be distinct and of same spelling

# command to see number of unique cities
df_silver.select("city").distinct().show()

In [0]:
#  cities name should be distinct and of same spelling
city_mapping = {
'Bengalurru' : 'Bengaluru',
'Bengalore' : 'Benglauru',
'Hyderbad' : 'Hyderabad',
'Hyderabadd' : 'Hyderabad',
'NewDelhee' : 'New Delhi',
'NewDheli' : 'New Delhi',
'NewDelhi' : 'New Delhi'
}

allowed = ['Bengaluru','Hyderabad','New Delhi']

df_silver = (
    df_silver
    .replace(city_mapping , subset=['city'])
    .withColumn(
        "city",
        F.when(F.col('city').isNull(), None)
        .when(F.col('city').isin(allowed), F.col('city'))
        .otherwise(None)
    )
)

In [0]:
# command to see number of unique cities
df_silver.select("city").distinct().show()

In [0]:
# capital small case fix in customer name 

# command to see unique customer names
df_silver.select("customer_name").distinct().show()

In [0]:
df_silver =  df_silver.withColumn(
    'customer_name',
    F.when(F.col('customer_name').isNull(), None)
    .otherwise(F.initcap('customer_name'))
)

In [0]:
# command to see unique customer names
df_silver.select("customer_name").distinct().show()

In [0]:
# customer having city null
df_silver.filter(F.col("city").isNull()).show(truncate=False)

In [0]:
# 
null_customer_names = ["Endurance Foods","Sprintx Nutrition","Zenathlete Foods","Primefuel Nutrition","Recovery Lane"]
df_silver.filter(F.col("customer_name").isin(null_customer_names)).show(truncate=False)

In [0]:
customer_city_fix = {
    789101 : "Bengaluru",
    789420 : "Bengaluru",
    789403 : "New Delhi",
    789520 : "Bengaluru",
    789521 : "Hyderabad",
    789603 : "Hyderabad"
}

df_fix = spark.createDataFrame(
    [(k, v) for k, v in customer_city_fix.items()],
    ["customer_id","fixed_city"]
)
display(df_fix)

In [0]:
df_silver = (
    df_silver
    .join(df_fix, "customer_id", "left")
    .withColumn(
        "city",
        F.coalesce("city","fixed_city")
    )
    .drop("fixed_city")
)
display(df_silver)

In [0]:
#customer id is number , convert it to string
df_silver = df_silver.withColumn("customer_id", F.col("customer_id").cast("string"))
df_silver.printSchema()

In [0]:
df_silver = (
    df_silver
    .withColumn(
        "customer",
        F.concat_ws("-","customer_name",F.coalesce(F.col("city"),F.lit("Unknown")))
    )
    .withColumn("market",F.lit("India"))
    .withColumn("platform",F.lit("Sports Bar"))
    .withColumn("channel",F.lit("Aquisition"))
)
display(df_silver.limit(5))

In [0]:
df_silver.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed","True")\
    .option("mergeSchema","True")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")

### Gold Processing 

In [0]:
df_silver = spark.sql(f"SELECT * FROM {catalog}.{silver_schema}.{data_source}")

df_gold = df_silver.select("customer_id","customer_name","city","customer","market","platform","channel")
display(df_gold)